In [26]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
customers_df = pd.read_csv(r"D:\Sasi\GUVI HCL AI-ML\Cart2Insights_Project\data\raw_datasets\olist_customers_dataset.csv")
orders_df = pd.read_csv(r"D:\Sasi\GUVI HCL AI-ML\Cart2Insights_Project\data\raw_datasets\olist_orders_dataset.csv")
products_df = pd.read_csv(r"D:\Sasi\GUVI HCL AI-ML\Cart2Insights_Project\data\raw_datasets\olist_products_dataset.csv")
sellers_df = pd.read_csv(r"D:\Sasi\GUVI HCL AI-ML\Cart2Insights_Project\data\raw_datasets\olist_sellers_dataset.csv")
order_payments_df = pd.read_csv(r"D:\Sasi\GUVI HCL AI-ML\Cart2Insights_Project\data\raw_datasets\olist_order_payments_dataset.csv")
order_reviews_df = pd.read_csv(r"D:\Sasi\GUVI HCL AI-ML\Cart2Insights_Project\data\raw_datasets\olist_order_reviews_dataset.csv")
order_items_df = pd.read_csv(r"D:\Sasi\GUVI HCL AI-ML\Cart2Insights_Project\data\raw_datasets\olist_order_items_dataset.csv")
product_category_transalation_df = pd.read_csv(r"D:\Sasi\GUVI HCL AI-ML\Cart2Insights_Project\data\raw_datasets\product_category_name_translation.csv")
geolocation_df = pd.read_csv(r"D:\Sasi\GUVI HCL AI-ML\Cart2Insights_Project\data\raw_datasets\olist_geolocation_dataset.csv")

### Handling missing values
**review_comment_title** and **review_comment_message** from **reviews_df** has high missing percentage values, but these are not mandatory to be filled since it is a choice made by customer to give only rating and not to leave comment.So, these can be changed to "Not provided".

In [27]:
order_reviews_df_c = order_reviews_df.copy()
columns_to_fill = ['review_comment_title', 'review_comment_message']
order_reviews_df_c[columns_to_fill] = order_reviews_df_c[columns_to_fill].fillna('Not provided')
order_reviews_df_c.isnull().sum()

review_id                  0
order_id                   0
review_score               0
review_comment_title       0
review_comment_message     0
review_creation_date       0
review_answer_timestamp    0
dtype: int64

Next, **order_delivered_customer_date**, **order_delivered_carrier_date**, **order_approved_at** from **orders_df** have significantly lower missing value percentage(~2.98% - 0.16%).So, they can simply be left as NaT as these values will be automatically ignored while performing time-series calculations.

In [28]:
orders_df.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

**product_name_lenght**, **product_category_name**, **product_description_lenght**, **product_photos_qty** from **products_df** have very low missing value percentage(1.85%).So, these can be replaced with respective median values except **product_category_name** which can be cahnged to 'Unknown'.

In [29]:
products_df_c = products_df.copy()

numeric_cols = ['product_name_lenght','product_photos_qty','product_description_lenght']
for col in numeric_cols:
    median_val = products_df_c[col].median()
    products_df_c.loc[:, col] = products_df_c.loc[:, col].fillna(products_df_c[col].fillna(median_val))

products_df_c['product_category_name'] = products_df_c['product_category_name'].fillna('Unknown')

products_df_c.isnull().sum()

product_id                    0
product_category_name         0
product_name_lenght           0
product_description_lenght    0
product_photos_qty            0
product_weight_g              2
product_length_cm             2
product_height_cm             2
product_width_cm              2
dtype: int64

**product_weight_g**, **product_length_cm**, **product_height_cm**, **product_width_cm** from **products_df** have only 2 missing values out of 32951 which is too low. So, these two rows can be dropped.

In [30]:
products_df_c = products_df_c.dropna()
products_df_c.isnull().sum()

product_id                    0
product_category_name         0
product_name_lenght           0
product_description_lenght    0
product_photos_qty            0
product_weight_g              0
product_length_cm             0
product_height_cm             0
product_width_cm              0
dtype: int64

### Removing duplicates
  The duplicate row are found only in **geolocation_df**. Since, the geolocation_zip_code_prefix column is going to be the primary key, its entries are made unique by grouping which eventually removes duplicates.

In [31]:
geolocation_df_c = geolocation_df.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat':'mean',
    'geolocation_lng':'mean',
    'geolocation_city':'first',
    'geolocation_state':'first'
}).reset_index()
display(geolocation_df.shape)
geolocation_df_c.shape

(1000163, 5)

(19015, 5)

### Formatting date/time columns:
   The date/time columns are found in **orders_df**, **order_reviews_df**, **order_items_df**, which are to be changed from str to date/time.

In [32]:
orders_df_c = orders_df.copy()
date_time_col = ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date','order_delivered_customer_date','order_estimated_delivery_date']
orders_df_c[date_time_col] = orders_df_c[date_time_col].apply(pd.to_datetime,errors='coerce')
display(orders_df_c.dtypes)
orders_df_c[orders_df_c.isnull().any(axis=1)]

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaT,NaT,2017-05-09
44,ee64d42b8cf066f35eac1cf57de1aa85,caded193e8e47b8362864762a83db3c5,shipped,2018-06-04 16:44:48,2018-06-05 04:31:18,2018-06-05 14:32:00,NaT,2018-06-28
103,0760a852e4e9d89eb77bf631eaaf1c84,d2a79636084590b7465af8ab374a8cf5,invoiced,2018-08-03 17:44:42,2018-08-07 06:15:14,NaT,NaT,2018-08-21
128,15bed8e2fec7fdbadb186b57c46c92f2,f3f0e613e0bdb9c7cee75504f0f90679,processing,2017-09-03 14:22:03,2017-09-03 14:30:09,NaT,NaT,2017-10-03
154,6942b8da583c2f9957e990d028607019,52006a9383bf149a4fb24226b173106f,shipped,2018-01-10 11:33:07,2018-01-11 02:32:30,2018-01-11 19:39:23,NaT,2018-02-07
...,...,...,...,...,...,...,...,...
99283,3a3cddda5a7c27851bd96c3313412840,0b0d6095c5555fe083844281f6b093bb,canceled,2018-08-31 16:13:44,NaT,NaT,NaT,2018-10-01
99313,e9e64a17afa9653aacf2616d94c005b8,b4cd0522e632e481f8eaf766a2646e86,processing,2018-01-05 23:07:24,2018-01-09 07:18:05,NaT,NaT,2018-02-06
99347,a89abace0dcc01eeb267a9660b5ac126,2f0524a7b1b3845a1a57fcf3910c4333,canceled,2018-09-06 18:45:47,NaT,NaT,NaT,2018-09-27
99348,a69ba794cc7deb415c3e15a0a3877e69,726f0894b5becdf952ea537d5266e543,unavailable,2017-08-23 16:28:04,2017-08-28 15:44:47,NaT,NaT,2017-09-15


In [33]:
order_reviews_df_c[['review_creation_date','review_answer_timestamp']] = order_reviews_df_c[['review_creation_date','review_answer_timestamp']].apply(pd.to_datetime,errors='coerce')
order_reviews_df_c.dtypes

review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

In [34]:
order_items_df_c = order_items_df.copy()
order_items_df_c['shipping_limit_date'] = pd.to_datetime(order_items_df_c['shipping_limit_date'],errors='coerce')
order_items_df_c.dtypes

order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object

### Handling invalid records

In [35]:
cond_1 = orders_df_c['order_approved_at'] < orders_df_c['order_purchase_timestamp']
cond_2 = orders_df_c['order_delivered_carrier_date'] < orders_df_c['order_approved_at']
cond_3 = orders_df_c['order_delivered_customer_date'] < orders_df_c['order_delivered_carrier_date']
invalid_dates = orders_df_c[cond_1 | cond_2 | cond_3]
invalid_dates

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
15,dcb36b511fcac050b97cd5c05de84dc3,3b6828a50ffe546942b7a473d70ac0fc,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00,2018-06-21 15:34:32,2018-07-04
64,688052146432ef8253587b930b01a06d,81e08b08e5ed4472008030d70327c71f,delivered,2018-04-22 08:48:13,2018-04-24 18:25:22,2018-04-23 19:19:14,2018-04-24 19:31:58,2018-05-15
199,58d4c4747ee059eeeb865b349b41f53a,1755fad7863475346bc6c3773fe055d3,delivered,2018-07-21 12:49:32,2018-07-26 23:31:53,2018-07-24 12:57:00,2018-07-25 23:58:19,2018-07-31
210,412fccb2b44a99b36714bca3fef8ad7b,c6865c523687cb3f235aa599afef1710,delivered,2018-07-22 22:30:05,2018-07-23 12:31:53,2018-07-23 12:24:00,2018-07-24 19:26:42,2018-07-31
415,56a4ac10a4a8f2ba7693523bb439eede,78438ba6ace7d2cb023dbbc81b083562,delivered,2018-07-22 13:04:47,2018-07-27 23:31:09,2018-07-24 14:03:00,2018-07-28 00:05:39,2018-08-06
...,...,...,...,...,...,...,...,...
99091,240ead1a7284667e0ec71d01f80e4d5e,fcdd7556401aaa1c980f8b67a69f95dc,delivered,2018-07-02 16:30:02,2018-07-05 16:17:59,2018-07-05 14:11:00,2018-07-10 23:21:47,2018-07-24
99230,78008d03bd8ef7fcf1568728b316553c,043e3254e68daf7256bda1c9c03c2286,delivered,2018-07-03 13:11:13,2018-07-05 16:32:52,2018-07-03 12:57:00,2018-07-10 17:47:39,2018-07-23
99266,76a948cd55bf22799753720d4545dd2d,3f20a07b28aa252d0502fe7f7eb030a9,delivered,2018-01-30 02:41:30,2018-02-04 23:31:46,2018-01-31 18:11:58,2018-03-18 20:08:50,2018-03-02
99377,a6bd1f93b7ff72cc348ca07f38ec4bee,6d63fa86bd2f62908ad328325799152f,delivered,2018-04-20 17:28:40,2018-04-24 19:26:10,2018-04-23 17:18:40,2018-04-28 17:38:42,2018-05-15


In [36]:
orders_df_c.loc[cond_1, 'order_approved_at'] = pd.NaT
orders_df_c.loc[cond_2, 'order_delivered_carrier_date'] = pd.NaT
orders_df_c.loc[cond_3, 'order_delivered_customer_date'] = pd.NaT
check_cond_1 = orders_df_c['order_approved_at'] < orders_df_c['order_purchase_timestamp']
check_cond_2 = orders_df_c['order_delivered_carrier_date'] < orders_df_c['order_approved_at']
check_cond_3 = orders_df_c['order_delivered_customer_date'] < orders_df_c['order_delivered_carrier_date']
re_check = orders_df_c[check_cond_1 | check_cond_2 | check_cond_3]
re_check

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


The date/time rows which doesn't match the chronological order of order purchased, approved and delivered are changed to NaT.

In [37]:
invalid_prices = order_items_df_c[(order_items_df_c['price']<0)|(order_items_df_c['freight_value']<0)]
invalid_prices

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


There is no negative value in price and freight value columns.

In [38]:
invalid_dim = products_df_c[(products_df_c['product_weight_g'] <= 0) | (products_df_c['product_length_cm'] <= 0)]
invalid_dim

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.0,529.0,1.0,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.0,528.0,1.0,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0


In [39]:
products_df_c['product_weight_g'] = products_df_c['product_weight_g'].replace(0.0, pd.NA)
products_df_c['product_weight_g'] = products_df_c['product_weight_g'].fillna(products_df_c.groupby('product_category_name')['product_weight_g'].transform('median'))
products_df_c[products_df_c['product_weight_g'] <= 0]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm


In product_weight column, the rows with 0 values are replaced with its category's median weight.

In [40]:
geolocation_df_c['geolocation_city'].value_counts()

geolocation_city
sao paulo          2652
são paulo           528
rio de janeiro      400
brasilia            393
salvador            275
                   ... 
charrua               1
agua santa            1
ciriaco               1
david canabarro       1
muliterno             1
Name: count, Length: 5955, dtype: int64

In [41]:
geolocation_df_c['geolocation_city'] = geolocation_df_c['geolocation_city'].str.lower().str.strip()
geolocation_df_c['geolocation_city'] = geolocation_df_c['geolocation_city'].str.normalize('NFKD').str.encode('ascii',errors='ignore').str.decode('utf-8')
geolocation_df_c['geolocation_city'].value_counts()

geolocation_city
sao paulo          3180
brasilia            494
rio de janeiro      400
salvador            275
goiania             236
                   ... 
charrua               1
agua santa            1
ciriaco               1
david canabarro       1
muliterno             1
Name: count, Length: 5771, dtype: int64

In [42]:
sellers_df_c['seller_city'] = sellers_df_c['seller_city'].str.lower().str.strip()
sellers_df_c['seller_city'] = sellers_df_c['seller_city'].str.normalize('NFKD').str.encode('ascii',errors='ignore').str.decode('utf-8')
sellers_df_c['seller_city'].value_counts()

seller_city
sao paulo                 695
curitiba                  127
rio de janeiro             96
belo horizonte             68
ribeirao preto             52
                         ... 
aparecida de goiania        1
bandeirantes                1
vitoria de santo antao      1
palotina                    1
leme                        1
Name: count, Length: 609, dtype: int64

In [43]:
customers_df_c['customer_city'] = customers_df_c['customer_city'].str.lower().str.strip()
customers_df_c['customer_city'] = customers_df_c['customer_city'].str.normalize('NFKD').str.encode('ascii',errors='ignore').str.decode('utf-8')
customers_df_c['customer_city'].value_counts()

customer_city
sao paulo              15540
rio de janeiro          6882
belo horizonte          2773
brasilia                2131
curitiba                1521
                       ...  
siriji                     1
natividade da serra        1
monte bonito               1
sao rafael                 1
eugenio de castro          1
Name: count, Length: 4119, dtype: int64

City names with special characters are corrected.

### Renaming columns

In [44]:
products_df_c = products_df_c.rename(columns={'product_name_lenght':'product_name_length',
                                              'product_description_lenght':'product_description_length'})
products_df_c.head()

,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


### Creating a copy of remaining datasets

In [45]:
customers_df_c = customers_df.copy()
sellers_df_c = sellers_df.copy()
order_payments_df_c = order_payments_df.copy()
product_category_transalation_df_c = product_category_transalation_df.copy()

In [46]:
import os
output_dir = r"D:\Sasi\GUVI HCL AI-ML\Cart2Insights_Project\data\cleaned_datasets"
datasets = {
    'customers': customers_df_c,
    'orders': orders_df_c,
    'products': products_df_c,
    'sellers': sellers_df_c,
    'order_payments': order_payments_df_c,
    'order_reviews': order_reviews_df_c,
    'order_items': order_items_df_c,
    'product_category_translation': product_category_transalation_df_c,
    'geolocation': geolocation_df_c
}

for name, df in datasets.items():
    file_path = os.path.join(output_dir,f"{name}.csv")
    df.to_csv(file_path ,index=False)
    print(f'Successfully saved : {file_path}')

Successfully saved : D:\Sasi\GUVI HCL AI-ML\Cart2Insights_Project\data\cleaned_datasets\customers.csv
Successfully saved : D:\Sasi\GUVI HCL AI-ML\Cart2Insights_Project\data\cleaned_datasets\orders.csv
Successfully saved : D:\Sasi\GUVI HCL AI-ML\Cart2Insights_Project\data\cleaned_datasets\products.csv
Successfully saved : D:\Sasi\GUVI HCL AI-ML\Cart2Insights_Project\data\cleaned_datasets\sellers.csv
Successfully saved : D:\Sasi\GUVI HCL AI-ML\Cart2Insights_Project\data\cleaned_datasets\order_payments.csv
Successfully saved : D:\Sasi\GUVI HCL AI-ML\Cart2Insights_Project\data\cleaned_datasets\order_reviews.csv
Successfully saved : D:\Sasi\GUVI HCL AI-ML\Cart2Insights_Project\data\cleaned_datasets\order_items.csv
Successfully saved : D:\Sasi\GUVI HCL AI-ML\Cart2Insights_Project\data\cleaned_datasets\product_category_translation.csv
Successfully saved : D:\Sasi\GUVI HCL AI-ML\Cart2Insights_Project\data\cleaned_datasets\geolocation.csv


In [47]:
datasets = {
    'customers': customers_df_c,
    'orders': orders_df_c,
    'products': products_df_c,
    'sellers': sellers_df_c,
    'order_payments': order_payments_df_c,
    'order_reviews': order_reviews_df_c,
    'order_items': order_items_df_c,
    'product_category_translation': product_category_transalation_df_c,
    'geolocation': geolocation_df_c
}

for name, df in datasets.items():
    display(df.shape)

(99441, 5)

(99441, 8)

(32949, 9)

(3095, 4)

(103886, 5)

(99224, 7)

(112650, 7)

(71, 2)

(19015, 5)